In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

FILE_PATH = "/content/drive/MyDrive/exposure_stochastic_benchmark.xlsx"
print("Loading benchmark sheets...")
df = pd.read_excel(FILE_PATH, sheet_name="results")

# Filter reference and adjacent single-step exposures
ref_df = df[df["Exposure"] == "REF"]
n1_df = df[df["Exposure"] == "_N1"]
p0_df = df[df["Exposure"] == "_P0"]

# 1. Skewness Threshold
# In balanced images, skewness stays within [-T_skew, +T_skew].
# We take the 90th percentile of absolute skewness in REF images.
ref_abs_skew = ref_df["skewness1"].abs()
t_skew_suggested = float(np.percentile(ref_abs_skew.dropna(), 90))

# 2. Outlier Margins
# Measure black and white counts on REF to see typical non-clipped levels
ref_blacks = ref_df["outliers1"].dropna()
ref_whites = ref_df["outliers2"].dropna()

print("\n--- RECOMMENDED THRESHOLDS ---")
print(f"Skewness Threshold (T_skew)   : {t_skew_suggested:.2f}")
print(f"Median REF Blacks (bins 0-4)  : {ref_blacks.median():.1f} pixels")
print(f"Median REF Whites (bins 251-255): {ref_whites.median():.1f} pixels")
print(f"95th Percentile REF Blacks    : {np.percentile(ref_blacks, 95):.1f} pixels")
print(f"95th Percentile REF Whites    : {np.percentile(ref_whites, 95):.1f} pixels")

Loading benchmark sheets...

--- RECOMMENDED THRESHOLDS ---
Skewness Threshold (T_skew)   : 56855.03
Median REF Blacks (bins 0-4)  : 0.0 pixels
Median REF Whites (bins 251-255): 0.0 pixels
95th Percentile REF Blacks    : 7.0 pixels
95th Percentile REF Whites    : 141.9 pixels


In [ ]:
import pandas as pd
import numpy as np

FILE_PATH = "/content/drive/MyDrive/exposure_stochastic_benchmark.xlsx"

print("Loading benchmark sheets...")
df = pd.read_excel(FILE_PATH, sheet_name="results")

def print_stats(exposure_label, column, name):
    data = df[df["Exposure"] == exposure_label][column].dropna()
    if len(data) == 0:
        return
    print(f"{exposure_label.ljust(5)} {name.ljust(6)} | "
          f"Median: {np.median(data):.1f} | "
          f"95th: {np.percentile(data, 95):.1f} | "
          f"99th: {np.percentile(data, 99):.1f} | "
          f"Max: {np.max(data):.1f}")

print("\n--- BLACK CLIPPING (Bins 0-4) ---")
for exp in ["REF", "_N1", "_N2", "_N3"]:
    print_stats(exp, "outliers1", "Blacks")

print("\n--- WHITE CLIPPING (Bins 251-255) ---")
for exp in ["REF", "_P1", "_P2", "_P3"]:
    print_stats(exp, "outliers2", "Whites")

Loading benchmark sheets...

--- BLACK CLIPPING (Bins 0-4) ---
REF   Blacks | Median: 0.0 | 95th: 7.0 | 99th: 75.7 | Max: 22195.0
_N1   Blacks | Median: 12.0 | 95th: 7520.5 | 99th: 21005.9 | Max: 53368.0
_N2   Blacks | Median: 149.0 | 95th: 15986.3 | 99th: 33211.6 | Max: 65536.0
_N3   Blacks | Median: 1145.0 | 95th: 28441.1 | 99th: 43390.1 | Max: 65536.0

--- WHITE CLIPPING (Bins 251-255) ---
REF   Whites | Median: 0.0 | 95th: 141.9 | 99th: 373.5 | Max: 1263.0
_P1   Whites | Median: 69.0 | 95th: 8222.0 | 99th: 19930.2 | Max: 45606.0
_P2   Whites | Median: 1392.0 | 95th: 19074.7 | 99th: 32300.3 | Max: 60497.0
_P3   Whites | Median: 8282.0 | 95th: 35343.6 | 99th: 50007.3 | Max: 65535.0


In [ ]:
import pandas as pd
import numpy as np

FILE_PATH = "/content/drive/MyDrive/exposure_stochastic_benchmark.xlsx"
df = pd.read_excel(FILE_PATH, sheet_name="results")

# Isolate the perfectly exposed reference images
ref_df = df[df["Exposure"] == "REF"]

def calc_3sigma_limit(series, name):
    mean = series.mean()
    std = series.std()
    limit = mean + (3 * std)
    print(f"{name.ljust(15)} | Mean: {mean:8.2f} | StdDev: {std:8.2f} | 3-Sigma Limit: {limit:8.2f}")

print("--- 3-SIGMA SAFE ZONES (99.7% Confidence for 0EV) ---")
calc_3sigma_limit(ref_df["outliers1"].dropna(), "Blacks (Bins 0-4)")
calc_3sigma_limit(ref_df["outliers2"].dropna(), "Whites (Bins 251-255)")
calc_3sigma_limit(ref_df["skewness1"].abs().dropna(), "Absolute Skew")

--- 3-SIGMA SAFE ZONES (99.7% Confidence for 0EV) ---
Blacks (Bins 0-4) | Mean:    14.32 | StdDev:   385.96 | 3-Sigma Limit:  1172.20
Whites (Bins 251-255) | Mean:    24.97 | StdDev:    76.14 | 3-Sigma Limit:   253.39
Absolute Skew   | Mean: 30258.32 | StdDev: 53424.96 | 3-Sigma Limit: 190533.21
